## 1. Focus on the stock you pick and their competitors. Report summary statistics of the training period you select and plot the kernel density. 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline
# For reading stock data from yahoo
from pandas_datareader.data import DataReader
import yfinance as yf
from pandas_datareader import data as pdr
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
yf.pdr_override()

# The tech stocks we'll use for this analysis
tech_list = ['PANW', 'FTNT', 'CHKP', 'CSCO']

# Set up End and Start times for data grab
start = '2015-01-01'
end= '2022-12-31'

for stock in tech_list:
    globals()[stock] = yf.download(stock, start, end)
    
company_list = [PANW, FTNT, CHKP, CSCO]
company_name = ["PALO ALTO NETWORKS", "FORTINET INC", "CHECKPOINT SOFTWARE", "CISCO"]

for company, com_name in zip(company_list, company_name):
    company["company_name"] = com_name
    
df = pd.concat(company_list, axis=0)
df

In [ ]:
# Let's see a historical view of the closing price
plt.figure(figsize=(20, 12))
plt.subplots_adjust(top=1.25, bottom=1.2)

for i, company in enumerate(company_list, 1):
    plt.subplot(2, 2, i)
    company['Adj Close'].plot()
    plt.ylabel('Adj Close')
    plt.xlabel(None)
    plt.title(f"Closing Price of {tech_list[i - 1]}")
    
plt.tight_layout()

In [ ]:
import yfinance as yf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define the tickers of the stocks to analyze
tickers = ['PANW', 'FTNT', 'CHKP', 'CSCO']

# Set the start and end dates for the training period
start_date = '2020-01-01'
end_date = '2022-12-31'

# Download the historical prices for each stock
prices = yf.download(tickers, start=start_date, end=end_date)['Adj Close']

# Compute the summary statistics for each stock
summary_stats = pd.concat([prices .mean(), prices .std(), prices .skew(), prices .kurtosis()], axis=1)
summary_stats.columns = ['Mean', 'Standard Deviation', 'Skewness', 'Kurtosis']
print(summary_stats)

# Plot the kernel density for each stock
plt.rcParams['figure.figsize'] = (20,6)
sns.kdeplot(prices ['PANW'], label='PANW')
sns.kdeplot(prices ['FTNT'], label='FTNT')
sns.kdeplot(prices ['CHKP'], label='CHKP')
sns.kdeplot(prices ['CSCO'], label='CSCO')
plt.legend()

In [ ]:
# Compute the daily returns for each stock
returns = prices.pct_change().dropna()

# Compute the summary statistics for each stock
summary_stats = pd.concat([returns.mean(), returns.std(), returns.skew(), returns.kurtosis()], axis=1)
summary_stats.columns = ['Mean', 'Standard Deviation', 'Skewness', 'Kurtosis']
print(summary_stats)

# Plot the kernel density for each stock
sns.kdeplot(returns['PANW'], label='PANW')
sns.kdeplot(returns['FTNT'], label='FTNT')
sns.kdeplot(returns['CHKP'], label='CHKP')
sns.kdeplot(returns['CSCO'], label='CSCO')
plt.legend()

## 2. Please use the features/factors you  take and discovered (e.g. FRED, Fama-French website, ADS, AR, CAPM, momentum factors, volume, price/return lags, etc.) to construct a feature database. The target variable Y can be either price or return. Frequency could be either daily or monthly.    

In [ ]:
# import required libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Get the historical stock data for Palo Alto Networks Inc
df = yf.download('PANW')
df.reset_index(inplace=True)

# filter the data for the required time range
start_date = '2015-01-01'
end_date = '2022-12-31'
df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

In [ ]:
# calculate the OBV values
df['OBV'] = np.where(df['Close'] > df['Close'].shift(1), df['Volume'], 
                     np.where(df['Close'] < df['Close'].shift(1), -df['Volume'], 0)).cumsum()

# create the lagged variables
df['Y_lag1'] = df['Open'].shift(1)
df['OBV_lag1'] = df['OBV'].shift(1)

df.drop(['OBV'],axis=1,inplace=True)

# drop NaN values
df = df.dropna()
df

In [ ]:
import os
from fredapi import Fred
import pandas as pd
fred = Fred(api_key=os.getenv('FRED_API_KEY'))
unrate = fred.get_series('UNRATE', observation_start='2015-01-01', observation_end='2022-12-31' )
df1 = pd.DataFrame({'Date': unrate.index, 'Unemployment Rate': unrate.values})

# Set the date as the index
df1.set_index('Date', inplace=True)

# Use the resample method to convert the data from monthly to daily
df1 = df1.resample('D').ffill()
df2= pd.merge(df, df1, on="Date")
df2

In [ ]:
cpia = fred.get_series('CPIAUCSL', observation_start='2015-01-01', observation_end='2022-12-31' )
df3 = pd.DataFrame({'Date': cpia.index, 'Consumer Index': cpia.values})

# Set the date as the index
df3.set_index('Date', inplace=True)

# Use the resample method to convert the data from monthly to daily
df3 = df3.resample('D').ffill()

df4= pd.merge(df2, df3, on="Date")
df4

In [ ]:
import pandas_datareader.data as web

# Define the start and end dates for the data
start_date_2 = '2015-01-01'
end_date_2 = '2022-12-31'

# Download the Fama-French daily data for the US market
ff_daily_us = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start=start_date_2, end=end_date_2)
ff_daily_us

# Extract the data from the downloaded object
ff_data = ff_daily_us[0]

# Convert the data to a Pandas DataFrame
ff_dataframe = ff_data.reset_index()

df5= pd.merge(df4, ff_dataframe)
df5['Date'] = df5['Date'].dt.strftime('%Y-%m-%d')
df5

In [ ]:
# Load ADS Index data
df_ads = pd.read_excel('ADS_Index_Most_Current_Vintage.xlsx')
df_ads = df_ads.rename(columns={df_ads.columns[0]: 'Date'})
df_ads['Date'] = df_ads['Date'].str.replace(':','-')
df_merge = pd.merge(df5, df_ads, on="Date")
df_merge

In [ ]:
df_merge.to_csv('feature_database.csv')

## 3. Virtualize the feature importance and feature selection process using regression based approach (Ridge regression, LASSO, Elastic Net or LARS) vs decision tree based approach (random forest, XGBoost). 

In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime
import numpy as np

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_data = df_merge[['Date','Y_lag1','OBV_lag1','Unemployment Rate','Consumer Index','Mkt-RF','SMB','HML','RF','ADS_Index','Adj Close']]

In [ ]:
# Convert 'Date' to DateTimeIndex
df_data['Date'] = pd.to_datetime(df_data['Date'])
df_data = df_data.set_index('Date')

In [ ]:
# Define the training period
train_start_date = '2015-01-01'
train_end_date = '2020-12-31'

# Define the testing period
test_start_date = '2021-01-01'
test_end_date = '2022-01-01'

# Split the data into training and testing sets
train = df_data.loc[train_start_date:train_end_date]
train = train.reset_index()
test = df_data.loc[test_start_date:test_end_date]
test = test.reset_index()
X_train = train.drop(columns = ['Date','Adj Close']).fillna(method='bfill')
y_train = np.diff(np.log(train['Adj Close'].values))
y_train = np.append(y_train[0], y_train)
X_test = test.drop(columns = ['Date','Adj Close']).fillna(method='bfill')
y_test = np.diff(np.log(test['Adj Close'].values))
y_test = np.append(y_test[0], y_test)

In [ ]:
from sklearn.linear_model import LinearRegression

# Create a linear regression model
model = LinearRegression()

### Ramdom Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
# define the model
rf_model = RandomForestRegressor(bootstrap=True, criterion='squared_error',
           max_features='auto',min_impurity_decrease=0.0,min_samples_leaf=1, min_samples_split=2,
           min_weight_fraction_leaf=0.0, n_estimators=100,random_state=0, verbose=0)

# fit the model
rf_model.fit(X_train, y_train)

In [ ]:
# 1. average feature importance
df_feature_importance = pd.DataFrame(model.feature_importances_, index=X_train.columns, 
                                     columns=['feature importance']).sort_values('feature importance', ascending=False)
print(df_feature_importance)

In [ ]:
plt.rcParams['figure.figsize'] = (20,12)
importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
plt.barh(range(X_train.shape[1]), importances[sorted_idx][::-1], align='center')
plt.yticks(range(X_train.shape[1]), X_train.columns[sorted_idx][::-1])
plt.title('Random Forest Feature Importance', fontsize=20)
plt.xlabel('Feature Importance', fontsize=16)
plt.ylabel('Feature' ,fontsize=16)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error as MSE
# make predictions for test data
y_pred = rf_model.predict(X_test)
  
# RMSE Computation
rmse = np.sqrt(MSE(y_test, y_pred))
print("RMSE : % f" %(rmse))

In [ ]:
df = pd.DataFrame(y_pred, columns=['y_pred'])
df2 = pd.DataFrame(y_test, columns=['y_test'])
df3 = pd.concat([df, df2], axis=1)
df3

In [ ]:
plt.figure(figsize=(20, 12))
plt.plot(df3.index, df3['y_test'], label='Actual Prices')
plt.plot(df3.index, df3['y_pred'], label='Predicted Prices')

### XGBoost

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
from sklearn.model_selection import GridSearchCV
# set up our search grid
param_grid = {"max_depth":    [5, 10],
              "n_estimators": [10, 50, 100],
              "learning_rate": [0.15, 0.3]}

# try out every combination of the above values
xgb_model = xgb.XGBRegressor(eval_metric='rmse')
search = GridSearchCV(regressor, param_grid, cv=5).fit(X_train, y_train)

print("The best hyperparameters are ",search.best_params_)
xgb_model = xgb.XGBRegressor(learning_rate = search.best_params_["learning_rate"],
                           n_estimators  = search.best_params_["n_estimators"],
                           max_depth     = search.best_params_["max_depth"],)

xgb_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error as MSE
# make predictions for test data
y_pred = xgb_model.predict(X_test)
  
# RMSE Computation
rmse = np.sqrt(MSE(y_test, y_pred))
print("RMSE : % f" %(rmse))

In [ ]:
df = pd.DataFrame(y_pred, columns=['y_pred'])
df2 = pd.DataFrame(y_test, columns=['y_test'])
df3 = pd.concat([df, df2], axis=1)
df3

In [ ]:
plt.figure(figsize=(20, 12))
plt.plot(df3.index, df3['y_test'], label='Actual Prices')
plt.plot(df3.index, df3['y_pred'], label='Predicted Prices')

In [ ]:
fig, ax = plt.subplots(figsize=(20,12))
plot_importance(xgb_model, ax=ax)
plt.title('XGBoost Feature Importance', fontsize=20)
plt.xlabel('Feature Importance', fontsize=16)
plt.ylabel('Feature' ,fontsize=16)
plt.show();

### Ridge Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn import linear_model
from sklearn.linear_model import Ridge
import statsmodels.api as sm
import scipy.stats as ss
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [ ]:
pipeline = Pipeline([('scaler',StandardScaler()),('model',Ridge())])
search = GridSearchCV(pipeline,{'model__alpha':np.arange(0.01,1,0.1)},cv = 5, scoring="neg_mean_squared_error",verbose=3
                      )
search.fit(X_train,y_train)
search.best_params_

In [ ]:
coefficients = search.best_estimator_.named_steps['model'].coef_
importance = np.abs(coefficients)
importance

In [ ]:
plt.bar(['Y_lag1','OBV_lag1','Unemployment Rate','Consumer Index','Mkt-RF','SMB','HML','RF','ADS_Index'], importance)
plt.show()

In [ ]:
# Factor Selection using Ridge Regression
a=0.91
model2_prep = linear_model.Ridge(alpha=a).fit(X_train, y_train)
model2_select = X_train.columns[np.abs(model2_prep.coef_)>=0.001]
x = X_train[model2_select]
model2 = sm.OLS(y_train,x).fit()
print(model2.summary())
y_pred_model2 = model2.predict(x)
corr_model2 = ss.pearsonr(y_pred_model2, y_train)[0]
print('model 2 Ridge Regression: corr (Y, Y_pred) = '+str(corr_model2))
print('Ridge Regression selected ' +str(len(model2_select)) +' features: ', model2_select.values)

In [ ]:
# ------------------------------------------------------------------------------
# Factor Selection using LASSO
# ------------------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

pipeline = Pipeline([
                     ('scaler',StandardScaler()),
                     ('model',Lasso())
])

search = GridSearchCV(pipeline,
                      {'model__alpha':np.arange(0.0001,0.01,0.0001)},
                      cv = 5, scoring="neg_mean_squared_error",verbose=3
                      )
search.fit(X_train,y_train)

search.best_params_

In [ ]:
coefficients = search.best_estimator_.named_steps['model'].coef_
importance = np.abs(coefficients)
importance

In [ ]:
plt.bar(['Y_lag1','OBV_lag1','Unemployment Rate','Consumer Index','Mkt-RF','SMB','HML','RF','ADS_Index'], importance)
plt.show()

In [ ]:
a = 0.0006
model3_prep = linear_model.Lasso(alpha=a).fit(X_train, y_train)
model3_select = X_train.columns[np.abs(model3_prep.coef_)>=0.001]
x = X_train[model3_select]
model3 = sm.OLS(y_train,x).fit()
print(model3.summary())
y_pred_model3 = model3.predict(x)
corr_model3 = ss.pearsonr(y_pred_model3, y_train)[0]
print('model 3 LASSO: corr (Y, Y_pred) = '+str(corr_model3))
print('LASSO selected ' +str(len(model3_select)) +' features: ', model3_select.values)

## 4. Proposed and train 3-6 models by feeding in the features you prepared in step 2.  Compare the model performance using RMSE between the fitted Y and actual Y in testing period. 

### LinearRegression

In [ ]:
from sklearn.linear_model import LinearRegression as LR
# Train the linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = lr.predict(X_test)

# Evaluate the model performance
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print("RMSE: ", rmse)

In [ ]:
df = pd.DataFrame(y_pred, columns=['y_pred'])
df2 = pd.DataFrame(y_test, columns=['y_test'])
df3 = pd.concat([df, df2], axis=1)
df3

In [ ]:
plt.figure(figsize=(20, 12))
plt.plot(df3.index, df3['y_test'], label='Actual Prices')
plt.plot(df3.index, df3['y_pred'], label='Predicted Prices')

### GradientBoostingRegressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor as GBR
reg2 = GBR(n_estimators=100,max_depth=5,max_features=20,random_state=0)
reg2.fit(X_train, y_train)
# Make predictions on the testing data
y_pred = reg2.predict(X_test)

# Evaluate the model performance
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print("RMSE: ", rmse)

In [ ]:
df = pd.DataFrame(y_pred, columns=['y_pred'])
df2 = pd.DataFrame(y_test, columns=['y_test'])
df3 = pd.concat([df, df2], axis=1)
df3

In [ ]:
plt.figure(figsize=(20, 12))
plt.plot(df3.index, df3['y_test'], label='Actual Prices')
plt.plot(df3.index, df3['y_pred'], label='Predicted Prices')